# Practical Project: Titanic Survival Prediction (Kaggle Style)

Perform a machine learning project from start to finish using a real dataset. Approach it as a Kaggle competition to learn practical skills.

**Learning Objectives:**
- Experience the complete ML workflow
- Perform Exploratory Data Analysis (EDA)
- Apply feature engineering techniques
- Compare and select multiple models
- Hyperparameter tuning
- Understand Kaggle competition strategies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Problem Definition

**Objective**: Develop a classification model to predict Titanic passenger survival

**Evaluation Metric**: Accuracy

**Data**: Passenger information including age, sex, cabin class, fare, etc.

## 2. Data Loading and Basic Exploration

In [ ]:
# Use seaborn's built-in Titanic dataset
# On Kaggle, you would download and use train.csv, test.csv
df = sns.load_dataset('titanic')

print("=== Basic Data Info ===")
print(f"Data shape: {df.shape}")
print(f"\nColumn list:")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Check first few rows
print("First 5 rows:")
df.head()

In [ ]:
# Descriptive statistics
print("Descriptive Statistics:")
df.describe()

In [ ]:
# Target variable distribution
print("=== Survival Distribution ===")
print(df['survived'].value_counts())
print(f"\nSurvival rate:")
print(df['survived'].value_counts(normalize=True))

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

df['survived'].value_counts().plot(kind='bar', ax=ax[0])
ax[0].set_title('Survival Count')
ax[0].set_xlabel('Survived (0=No, 1=Yes)')
ax[0].set_ylabel('Count')

df['survived'].value_counts(normalize=True).plot(kind='pie', autopct='%1.1f%%', ax=ax[1])
ax[1].set_title('Survival Proportion')
ax[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Missing Value Analysis

In [ ]:
# Check missing values
print("=== Missing Value Analysis ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Ratio (%)': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))

# Visualization
plt.figure(figsize=(10, 6))
missing_data = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
plt.barh(missing_data.index, missing_data['Missing Ratio (%)'])
plt.xlabel('Missing Percentage (%)')
plt.title('Missing Values by Feature')
plt.tight_layout()
plt.show()

### 3.2 Relationship Between Categorical Variables and Survival

In [ ]:
# Relationship between key categorical variables and survival
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Sex
sns.countplot(data=df, x='sex', hue='survived', ax=axes[0, 0])
axes[0, 0].set_title('Survival by Sex')

# Cabin class
sns.countplot(data=df, x='pclass', hue='survived', ax=axes[0, 1])
axes[0, 1].set_title('Survival by Class')

# Embarkation port
sns.countplot(data=df, x='embarked', hue='survived', ax=axes[0, 2])
axes[0, 2].set_title('Survival by Embarked')

# Siblings/spouses count
sns.countplot(data=df, x='sibsp', hue='survived', ax=axes[1, 0])
axes[1, 0].set_title('Survival by SibSp')

# Parents/children count
sns.countplot(data=df, x='parch', hue='survived', ax=axes[1, 1])
axes[1, 1].set_title('Survival by Parch')

# Traveling alone
df['alone'] = ((df['sibsp'] + df['parch']) == 0).astype(int)
sns.countplot(data=df, x='alone', hue='survived', ax=axes[1, 2])
axes[1, 2].set_title('Survival by Alone')

plt.tight_layout()
plt.show()

In [ ]:
# Survival rate statistics
print("=== Survival Rate by Category ===")
print("\nBy sex:")
print(df.groupby('sex')['survived'].mean())
print("\nBy cabin class:")
print(df.groupby('pclass')['survived'].mean())
print("\nBy embarkation port:")
print(df.groupby('embarked')['survived'].mean())

### 3.3 Numeric Variable Analysis

In [ ]:
# Age and fare distributions (by survival)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution
for survived in [0, 1]:
    axes[0, 0].hist(df[df['survived'] == survived]['age'].dropna(), 
                    bins=30, alpha=0.5, label=f'Survived={survived}')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Age Distribution by Survival')
axes[0, 0].legend()

# Age boxplot
sns.boxplot(data=df, x='survived', y='age', ax=axes[0, 1])
axes[0, 1].set_title('Age by Survival')

# Fare distribution (log scale)
for survived in [0, 1]:
    axes[1, 0].hist(np.log1p(df[df['survived'] == survived]['fare'].dropna()), 
                    bins=30, alpha=0.5, label=f'Survived={survived}')
axes[1, 0].set_xlabel('Log(Fare + 1)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Fare Distribution by Survival (Log Scale)')
axes[1, 0].legend()

# Fare boxplot
sns.boxplot(data=df, x='survived', y='fare', ax=axes[1, 1])
axes[1, 1].set_title('Fare by Survival')
axes[1, 1].set_ylim(0, 300)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
print("=== Numeric Variable Correlations ===")
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nCorrelation with target (survived):")
print(correlation['survived'].sort_values(ascending=False))

## 4. Data Preprocessing and Feature Engineering

In [ ]:
# Copy data for processing
df_clean = df.copy()

print("=== Preprocessing Start ===")
print(f"Initial data shape: {df_clean.shape}")

### 4.1 Remove Unnecessary Columns

In [ ]:
# Remove redundant or unnecessary columns
drop_cols = ['deck', 'embark_town', 'alive', 'who', 'adult_male', 'class']
df_clean = df_clean.drop(columns=drop_cols, errors='ignore')

print(f"After column removal: {df_clean.shape}")
print(f"Remaining columns: {df_clean.columns.tolist()}")

### 4.2 Missing Value Handling

In [ ]:
# Age: fill with median
age_median = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(age_median)
print(f"Age missing values filled with median ({age_median})")

# Embarkation port: fill with mode
embarked_mode = df_clean['embarked'].mode()[0]
df_clean['embarked'] = df_clean['embarked'].fillna(embarked_mode)
print(f"Embarked missing values filled with mode ({embarked_mode})")

# Fare: fill with median
fare_median = df_clean['fare'].median()
df_clean['fare'] = df_clean['fare'].fillna(fare_median)

print(f"\nAfter missing value handling:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

### 4.3 Feature Engineering

Create new features using domain knowledge.

In [ ]:
# 1. Family size
# Why: Family size is a compound feature that captures survival dynamics better than
# sibsp/parch alone — medium-sized families (2-4) had higher survival rates because
# they could help each other, while very large families struggled.
df_clean['family_size'] = df_clean['sibsp'] + df_clean['parch'] + 1
print("Family size feature created: sibsp + parch + 1")

# 2. Traveling alone
df_clean['is_alone'] = (df_clean['family_size'] == 1).astype(int)
print("Traveling alone feature created")

# 3. Age group
# Why: Binning continuous age into groups captures non-linear survival patterns
# (e.g., children had priority boarding on lifeboats) that linear models miss.
df_clean['age_group'] = pd.cut(df_clean['age'],
                                bins=[0, 12, 18, 35, 60, 100],
                                labels=['Child', 'Teen', 'Young', 'Middle', 'Senior'])
print("Age group feature created")

# 4. Fare bin
df_clean['fare_bin'] = pd.qcut(df_clean['fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
print("Fare bin feature created")

# 5. Title extraction (optional)
# df_clean['title'] = df_clean['name'].str.extract(' ([A-Za-z]+)\.', expand=False)

print(f"\nShape after feature engineering: {df_clean.shape}")

In [ ]:
# Check relationship between new features and survival
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(data=df_clean, x='family_size', hue='survived', ax=axes[0])
axes[0].set_title('Survival by Family Size')

sns.countplot(data=df_clean, x='age_group', hue='survived', ax=axes[1])
axes[1].set_title('Survival by Age Group')
axes[1].tick_params(axis='x', rotation=45)

sns.countplot(data=df_clean, x='fare_bin', hue='survived', ax=axes[2])
axes[2].set_title('Survival by Fare Bin')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4.4 Categorical Variable Encoding

In [ ]:
# LabelEncoder
le = LabelEncoder()

df_clean['sex'] = le.fit_transform(df_clean['sex'])
df_clean['embarked'] = le.fit_transform(df_clean['embarked'])
df_clean['age_group'] = le.fit_transform(df_clean['age_group'])
df_clean['fare_bin'] = le.fit_transform(df_clean['fare_bin'])

print("Categorical variable encoding complete")
print(f"\nData types after encoding:")
print(df_clean.dtypes)

### 4.5 Final Feature Selection

In [ ]:
# Select features for modeling
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
            'embarked', 'family_size', 'is_alone', 'age_group', 'fare_bin']

X = df_clean[features]
y = df_clean['survived']

print(f"Final features: {features}")
print(f"X shape: {X.shape}")
print(f"y distribution: {y.value_counts().to_dict()}")

## 5. Modeling

### 5.1 Data Splitting

In [ ]:
# Why: stratify=y ensures the train/test split preserves the original class ratio (~38% survived).
# Without stratification, small datasets risk an unrepresentative split that biases evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")
print(f"\nTraining target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")

In [ ]:
# Scaling (for linear models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")

### 5.2 Baseline Model

Establish a baseline with a simple model.

In [ ]:
# Baseline: always predict majority class
baseline_pred = np.zeros(len(y_test))  # Predict all 0 (deceased)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Baseline accuracy (always predict deceased): {baseline_acc:.4f}")
print("\nWe aim for performance above this value.")

### 5.3 Comparing Multiple Models

In [ ]:
# Define various models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

# Model comparison
print("=== Model Comparison (5-Fold Cross Validation) ===")
results = []

for name, model in models.items():
    # Linear models use scaled data
    if name in ['Logistic Regression', 'SVM']:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='accuracy')
    
    # Train and test
    model.fit(X_tr, y_train)
    test_score = model.score(X_te, y_test)
    
    results.append({
        'Model': name,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Score': test_score
    })
    
    print(f"{name}:")
    print(f"  CV = {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Test = {test_score:.4f}")
    print()

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='CV Mean', ascending=False)
print("\nModel ranking:")
print(results_df)

In [ ]:
# Results visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV scores
axes[0].barh(results_df['Model'], results_df['CV Mean'])
axes[0].set_xlabel('CV Accuracy')
axes[0].set_title('Cross-Validation Scores')
axes[0].set_xlim(0.7, 0.9)

# Test scores
axes[1].barh(results_df['Model'], results_df['Test Score'])
axes[1].set_xlabel('Test Accuracy')
axes[1].set_title('Test Scores')
axes[1].set_xlim(0.7, 0.9)

plt.tight_layout()
plt.show()

### 5.4 Hyperparameter Tuning

Tune hyperparameters for the best performing model.

In [ ]:
# Random Forest tuning
# Why: This grid covers the key axes of RF complexity — n_estimators controls ensemble
# size, max_depth limits tree depth to prevent overfitting, and min_samples_split/leaf
# regulate how aggressively trees partition the data.
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    rf, rf_param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=1
)

print("Grid Search starting...")
grid_search.fit(X_train, y_train)

print("\n=== Hyperparameter Tuning Results ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Test score: {grid_search.score(X_test, y_test):.4f}")

best_model = grid_search.best_estimator_

## 6. Model Evaluation

### 6.1 Classification Performance Metrics

In [ ]:
# Prediction
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Classification report
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

# ROC AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC AUC Score: {roc_auc:.4f}")

### 6.2 Confusion Matrix

In [ ]:
# Confusion matrix visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'],
            ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 6.3 Feature Importance

In [ ]:
# Feature importance
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [features[i] for i in indices], rotation=45)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

print("\nFeature importance ranking:")
for i in indices:
    print(f"  {features[i]:15s}: {importances[i]:.4f}")

### 6.4 Error Analysis

In [ ]:
# Analyze misclassified cases
X_test_df = X_test.copy()
X_test_df['actual'] = y_test.values
X_test_df['predicted'] = y_pred
X_test_df['correct'] = X_test_df['actual'] == X_test_df['predicted']

print("=== Prediction Results ===")
print(f"Correctly predicted: {X_test_df['correct'].sum()} / {len(X_test_df)}")
print(f"Incorrectly predicted: {(~X_test_df['correct']).sum()} / {len(X_test_df)}")

# False Positive and False Negative
fp = X_test_df[(X_test_df['actual'] == 0) & (X_test_df['predicted'] == 1)]
fn = X_test_df[(X_test_df['actual'] == 1) & (X_test_df['predicted'] == 0)]

print(f"\nFalse Positive (actual deceased, predicted survived): {len(fp)}")
print(f"False Negative (actual survived, predicted deceased): {len(fn)}")

print("\nFalse Negative samples (first 5):")
print(fn.head())

## 7. Kaggle Competition Strategies

### 7.1 Ensemble Techniques

In [ ]:
# Why: Blending averages probability predictions from diverse models, reducing variance
# through model diversity — different algorithms make different errors, so their average
# is often more reliable than any single model's prediction.
def simple_blend(models, X_train, y_train, X_test, weights=None):
    """Simple blending ensemble"""
    if weights is None:
        weights = [1/len(models)] * len(models)
    
    predictions = np.zeros(len(X_test))
    
    for model, weight in zip(models, weights):
        model.fit(X_train, y_train)
        pred_proba = model.predict_proba(X_test)[:, 1]
        predictions += weight * pred_proba
    
    return (predictions > 0.5).astype(int)


# Ensemble models
ensemble_models = [
    RandomForestClassifier(n_estimators=200, random_state=42),
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    LogisticRegression(max_iter=1000, random_state=42)
]

# Third model uses scaled data
y_pred_ensemble = simple_blend(
    [ensemble_models[0], ensemble_models[1]], 
    X_train, y_train, X_test
)

# Evaluation
ensemble_acc = accuracy_score(y_test, y_pred_ensemble)
print(f"Ensemble accuracy: {ensemble_acc:.4f}")
print(f"Best single model accuracy: {best_model.score(X_test, y_test):.4f}")
print(f"Improvement: {(ensemble_acc - best_model.score(X_test, y_test)):.4f}")

### 7.2 Kaggle Submission File Format

In [ ]:
# Generate predictions for Kaggle submission (in actual Kaggle, use test.csv)
# Here we use test data as an example

submission = pd.DataFrame({
    'PassengerId': range(1, len(y_pred) + 1),  # Actually use PassengerId from test.csv
    'Survived': y_pred
})

print("Submission file format:")
print(submission.head(10))

# Save to CSV
# submission.to_csv('titanic_submission.csv', index=False)
# print("\nsubmission.csv saved")

## 8. Essential Kaggle Tips

### 8.1 Competition Checklist

**1. Quick Start**
- Run baseline code and make first submission
- Check leaderboard position

**2. Focus on EDA**
- Understanding data is key
- Identify missing values, outliers, distributions
- Analyze relationship with target

**3. Feature Engineering**
- Leverage domain knowledge
- Create cross-features (e.g., family_size)
- Group-level statistics (e.g., group means)

**4. Try Various Models**
- Linear models -> Tree-based -> Ensemble
- Hyperparameter tuning

**5. Ensemble**
- Combine predictions from different models
- Blending, Stacking

**6. Validation Strategy**
- Verify local CV matches leaderboard score
- Watch out for overfitting (don't overfit to Public LB)

### 8.2 Cross-Validation Strategy

In [ ]:
def cross_validate_model(model, X, y, n_splits=5, stratified=True):
    """
    Perform cross-validation
    
    Parameters:
    -----------
    model : sklearn estimator
    X : features
    y : target
    n_splits : number of folds
    stratified : whether to use stratification
    """
    # Why: StratifiedKFold preserves the class distribution in each fold, which is
    # essential for imbalanced or small datasets to ensure each fold is representative.
    if stratified:
        kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    else:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]
        
        model.fit(X_train_fold, y_train_fold)
        score = model.score(X_val_fold, y_val_fold)
        scores.append(score)
        
        print(f"Fold {fold+1}: {score:.4f}")
    
    print(f"\nMean: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")
    return np.mean(scores)


# Usage example
print("=== Random Forest Cross-Validation ===")
cv_score = cross_validate_model(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X, y, n_splits=5
)

## Summary

### Project Workflow

1. **Problem Definition**: Set objectives and evaluation metrics
2. **Data Exploration**: Understand data through EDA
3. **Preprocessing**: Handle missing values, encoding
4. **Feature Engineering**: Leverage domain knowledge
5. **Modeling**: Compare multiple models
6. **Tuning**: Hyperparameter optimization
7. **Evaluation**: Assess performance with various metrics
8. **Ensemble**: Combine multiple models

### Key Points

- **EDA is most important**: Good models cannot be built without understanding the data
- **Feature engineering**: The key to improving model performance
- **Cross-validation**: Prevents overfitting and confirms generalization performance
- **Ensemble**: Improve performance by combining diverse models
- **Iterative improvement**: No perfect model on the first try, continuous improvement is needed